In [ ]:


# ==============================
# Imports
# ==============================
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import cv2
import matplotlib.pyplot as plt
import pennylane as qml

# ==============================
# LULC Class Definition
# ==============================
LULC_CLASSES = {
    0: "Built-up",
    1: "Vegetation",
    2: "Water bodies",
    3: "Others land"
}

CLASS_COLORS = {
    0: [255, 0, 0],     # Built-up (Red)
    1: [0, 255, 0],     # Vegetation (Green)
    2: [0, 0, 255],     # Water (Blue)
    3: [255, 255, 0],   # Others (Yellow)
}

NUM_CLASSES = 4
PATCH_SIZE = 128

# ==============================
# Classical CNN (Baseline)
# ==============================
class ClassicalCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.conv3 = nn.Conv2d(64, 128, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(16 * 16 * 128, 256)
        self.fc2 = nn.Linear(256, num_classes)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = x.view(x.size(0), -1)
        x = F.relu(self.fc1(x))
        return F.softmax(self.fc2(x), dim=1)

# ==============================
# Quantum Circuit Definition
# ==============================
n_qubits = 4
dev = qml.device("default.qubit", wires=n_qubits)

@qml.qnode(dev)
def quantum_circuit(inputs, weights):
    # ZFeatureMap-like encoding
    for i in range(n_qubits):
        qml.Hadamard(wires=i)
        qml.RZ(2 * inputs[i], wires=i)
        qml.Hadamard(wires=i)
        qml.RZ(2 * inputs[i], wires=i)

    # Quantum convolution (RX–RY–RZ)
    for i in range(n_qubits):
        qml.RX(weights[i], wires=i)
        qml.RY(weights[i + 4], wires=i)
        qml.RZ(weights[i + 8], wires=i)

    # Local entanglement
    qml.CNOT(wires=[0, 1])
    qml.CNOT(wires=[2, 3])

    return [qml.expval(qml.PauliZ(i)) for i in range(n_qubits)]

# ==============================
# QCNN Model
# ==============================
class QCNN(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.q_params = nn.Parameter(torch.randn(12))
        self.fc = nn.Linear(4, num_classes)

    def forward(self, x):
        outputs = []
        for sample in x:
            # Reduce patch to 4 features (mean spectral values)
            inputs = sample.mean(dim=(1, 2))[:4]
            q_out = quantum_circuit(
                inputs.detach().numpy(),
                self.q_params.detach().numpy()
            )
            outputs.append(q_out)
        q_features = torch.tensor(outputs, dtype=torch.float32)
        return F.softmax(self.fc(q_features), dim=1)

# ==============================
# Hybrid QCNN Model
# ==============================
class HybridQCNN(QCNN):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__(num_classes)
        self.fc1 = nn.Linear(4, 32)
        self.fc2 = nn.Linear(32, num_classes)

    def forward(self, x):
        q_features = super().forward(x)
        x = F.relu(self.fc1(q_features))
        return F.softmax(self.fc2(x), dim=1)

# ==============================
# Patch-wise Inference
# ==============================
def predict_lulc_map(model, image):
    h, w, _ = image.shape
    label_map = np.zeros((h, w), dtype=np.uint8)

    for i in range(0, h, PATCH_SIZE):
        for j in range(0, w, PATCH_SIZE):
            patch = image[i:i+PATCH_SIZE, j:j+PATCH_SIZE]
            if patch.shape[:2] != (PATCH_SIZE, PATCH_SIZE):
                continue
            patch = patch / 255.0
            patch_tensor = torch.tensor(patch).permute(2, 0, 1).unsqueeze(0).float()
            with torch.no_grad():
                pred = model(patch_tensor).argmax(dim=1).item()
            label_map[i:i+PATCH_SIZE, j:j+PATCH_SIZE] = pred

    return label_map

# ==============================
# Color Mapping
# ==============================
def colorize_map(label_map):
    h, w = label_map.shape
    color_map = np.zeros((h, w, 3), dtype=np.uint8)
    for cls, color in CLASS_COLORS.items():
        color_map[label_map == cls] = color
    return color_map

# ==============================
# Main Execution
# ==============================
if __name__ == "__main__":

    # Load sample input image
    img = cv2.imread("input_liss3.png")  # replace with your image
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Initialize models (load trained weights if available)
    cnn_model = ClassicalCNN()
    qcnn_model = QCNN()
    hybrid_model = HybridQCNN()

    # Inference
    cnn_map = predict_lulc_map(cnn_model, img)
    qcnn_map = predict_lulc_map(qcnn_model, img)
    hybrid_map = predict_lulc_map(hybrid_model, img)

    # Colorize outputs
    cnn_color = colorize_map(cnn_map)
    qcnn_color = colorize_map(qcnn_map)
    hybrid_color = colorize_map(hybrid_map)

    # Visualization (matches Figure 11)
    plt.figure(figsize=(10, 8))

    plt.subplot(2, 2, 1)
    plt.imshow(img)
    plt.title("(a) Multispectral Input")
    plt.axis("off")

    plt.subplot(2, 2, 2)
    plt.imshow(cnn_color)
    plt.title("(b) Classical CNN")
    plt.axis("off")

    plt.subplot(2, 2, 3)
    plt.imshow(qcnn_color)
    plt.title("(c) QCNN")
    plt.axis("off")

    plt.subplot(2, 2, 4)
    plt.imshow(hybrid_color)
    plt.title("(d) Hybrid QCNN")
    plt.axis("off")

    plt.tight_layout()
    plt.show()
